[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/04-recall-tuning/01-understanding_match_modes.ipynb)

# Understanding Match Modes

Every query you've run so far used the default matching behavior: fuzzy, typo-tolerant comparison. That default is called `APPROX`, and it's one of four string comparison strategies M|BOX supports. Each one answers a genuinely different question about how two strings relate to each other, and picking the wrong one for a given field can either let through matches you don't want, or silently reject matches you do.

In this notebook you will:

1. See `APPROX` explicitly, the mode you've been using all along
2. See `EXACT`, and why forcing strict equality is sometimes exactly what you want
3. See `COMPLETE`, for when your query is a fragment expected inside a longer indexed value
4. See `DETECT`, the reverse case, when the indexed value is a fragment expected inside a longer query
5. Compare all four side by side on the same pair of strings
6. Know which mode fits which situation

In [1]:
# !pip install mbox

## Setting up

Here's a small product catalog we'll use throughout this notebook, loaded from `datasets/product_catalog.csv`, with a longer free-text description field alongside the usual identifier and name.

In [2]:
import pandas as pd
from mbox.indexing import TableIndexer

df = pd.read_csv("datasets/product_catalog.csv")

index = TableIndexer.create_index(
    df=df,
    index_columns=["product_id", "product_name", "description"],
    tmp_dir="tmp_index"
)

index

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object


## 1. `APPROX`: fuzzy, typo-tolerant comparison

`APPROX` is the mode you've been using implicitly this whole cookbook. It measures how close two strings are, character by character, and produces a continuous score rather than a strict pass or fail. It's the right default for anything a human might type slightly wrong, names, free text, descriptions.

In [3]:
from mbox.recall import TableRecallMode

approx_result = index.match(
    product_name="Extendd Battery Pak",
    modes={"product_name": TableRecallMode.APPROX},
    include_field_scores=True
)

approx_result

,query_row,index_row,product_name_candidate,product_id_candidate,description_candidate,overall_score,product_name_score
0,0,0,Extended Battery Pack,B88-EXT,Extended battery pack for outdoor and camping use,75,75


Even with two typos in the query, `product_name_score` should still land reasonably high. This is the entire point of `APPROX`: small, human-sized mistakes shouldn't tank a match.

## 2. `EXACT`: strict full equality

Sometimes fuzziness is a liability rather than a convenience. If you're matching against a legal identifier, a transaction reference, or anything else where "close" is meaningfully different from "correct," you want `EXACT`: a full match scores 100, anything else scores 0. No partial credit.

In [4]:
exact_correct = index.match(
    product_id="B88-EXT",
    modes={"product_id": TableRecallMode.EXACT},
    include_field_scores=True
)

exact_typo = index.match(
    product_id="B88-EXY",   # one character off
    modes={"product_id": TableRecallMode.EXACT},
    min_total_match_value=0,   # keep the low-scoring result visible for comparison
    include_field_scores=True
)

print("Exact match:")
display(exact_correct)

print("\nOne character off, under EXACT mode:")
display(exact_typo)

Exact match:


,query_row,index_row,product_id_candidate,product_name_candidate,description_candidate,overall_score,product_id_score
0,0,0,B88-EXT,Extended Battery Pack,Extended battery pack for outdoor and camping use,100,100



One character off, under EXACT mode:


,query_row,index_row,product_id_candidate,product_name_candidate,description_candidate,overall_score,product_id_score
0,0,-1,,,,0,0


The correct `product_id` should score 100, and the one-character-off version should score 0, `product_id_score` doesn't degrade gracefully here the way it would under `APPROX`. That's a deliberate, useful property: for a field like a product ID, you often want the system to tell you plainly "no, this isn't a match" rather than offering a confident-looking but wrong candidate.

## 3. `COMPLETE`: your query is a fragment inside a longer value

`COMPLETE` checks whether the input value exists as a substring within the indexed value. This fits searches where someone types a short keyword and expects to find it inside a longer piece of text, the way a site search box behaves.

In [5]:
complete_result = index.match(
    description="battery pack",
    modes={"description": TableRecallMode.COMPLETE},
    include_field_scores=True
)

complete_result

,query_row,index_row,description_candidate,product_id_candidate,product_name_candidate,overall_score,description_score
0,0,0,Extended battery pack for outdoor and camping use,B88-EXT,Extended Battery Pack,85,85


`"battery pack"` is a short fragment of the full indexed description, `"Extended battery pack for outdoor and camping use"`, and `COMPLETE` should find it. Try the same query under `APPROX` and compare the difference in how confidently it resolves, since `APPROX` is measuring overall character similarity across the whole string, not fragment containment specifically.

## 4. `DETECT`: the indexed value is a fragment inside a longer query

`DETECT` is the mirror image of `COMPLETE`: it checks whether the *indexed* value exists as a substring within the *query*. This fits situations where your query is a longer piece of free text, a support ticket, a customer message, a search log entry, and you're checking whether it mentions something specific and short that you already have indexed.

In [6]:
detect_result = index.match(
    product_name="I need an extended battery pack for my camping trip next week",
    modes={"product_name": TableRecallMode.DETECT},
    include_field_scores=True
)

detect_result

,query_row,index_row,product_name_candidate,product_id_candidate,description_candidate,overall_score,product_name_score
0,0,0,Extended Battery Pack,B88-EXT,Extended battery pack for outdoor and camping use,85,85


Here, the query is a full sentence, and the short indexed value `"Extended Battery Pack"` should be found inside it. `COMPLETE` would have looked for the entire sentence inside the product name and failed; `DETECT` is built for exactly the reversed direction.

## 5. All four modes, side by side

To make the difference concrete, let's run the exact same query and indexed pair through all four modes and compare the scores directly.

In [7]:
for mode in [TableRecallMode.APPROX, TableRecallMode.EXACT, TableRecallMode.COMPLETE, TableRecallMode.DETECT]:
    result = index.match(
        product_name="Extended Battery",
        modes={"product_name": mode},
        min_total_match_value=0,
        include_field_scores=True
    )
    top_score = result["product_name_score"].iloc[0] if len(result) else "no match returned"
    print(f"{mode.value:10s} -> {top_score}")

Approx     -> 96
Exact      -> 0
Complete   -> 96
Detect     -> 46


`"Extended Battery"` is a fragment of the full indexed name `"Extended Battery Pack"`. You should see something like:

- `APPROX` scores high, but not perfect, most of the string overlaps, but it's not identical
- `EXACT` scores 0, the strings aren't fully equal
- `COMPLETE` scores high, the query is genuinely contained within the indexed value
- `DETECT` scores low or 0, the *indexed* value is not contained within the shorter query, it's the other way around

Same input, same indexed data, four different, valid answers, depending on what question you're actually asking.

## A note on numeric fields

Everything above applies to string fields. Numeric fields have their own parallel set of modes, `NUM_APPROX`, `NUM_EXACT`, `NUM_GREATER`, `NUM_LOWER`, for comparing quantities, prices, and dates by proximity or range rather than by character overlap. Those get a full notebook of their own next: `03-numeric_matching_ranges_and_thresholds.ipynb`.

## 6. Choosing a mode

| If you're matching... | Use |
|---|---|
| Free text a human typed, names, addresses, descriptions | `APPROX` |
| Identifiers, codes, or anything where "close" should not count | `EXACT` |
| A short keyword that should be found inside a longer indexed value | `COMPLETE` |
| A longer query that might mention a shorter, already-indexed term | `DETECT` |

You can set a mode per field, either inline with `modes={}` on `match()` as we've done throughout this notebook, or as part of a full `TableRecallFieldConfig`, covered in the next notebook, `02-weighting_fields_for_better_precision.ipynb`.

## Next steps

- **`02-weighting_fields_for_better_precision.ipynb`** - assign different importance to different fields, and set minimum quality thresholds per field
- **`03-numeric_matching_ranges_and_thresholds.ipynb`** - the numeric equivalents of what you just learned
- **`04-reusable_recall_configs_as_json.ipynb`** - save a fully tuned recall configuration and reuse it across a pipeline

*M|BOX is currently in `beta`. Breaking changes may occur in minor releases until version `1.0.0`.*